# Notebook 05: Hensen Open Data Audit

This notebook performs a detailed audit of the Hensen et al. (2015) Delft loophole-free Bell-test dataset. 
It verifies the loading process, filtering steps, and reproduction of the published CHSH result.

In [2]:
from __future__ import annotations
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

# Standardized project root addition
project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from xtheta.data.adapters.hensen import load_hensen_dataset
from xtheta.data.validation import run_open_data_chsh_validation

## 1. Load Raw Data

We load the raw data to see the initial row count and inspect the first few lines.

In [3]:
data_path = project_root / "data" / "open_bell" / "hensen" / "raw" / "bell_open_data.txt"
df_raw = pd.read_csv(data_path, header=None)
print(f"Raw row count: {len(df_raw)}")
df_raw.head()

Raw row count: 4746


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,2015-06-26 17:24:12.119993,1,2,5445065,1,5671004,1,1,1,10379,10371,11281,13113,0,0,0,0
1,2015-06-26 17:29:29.620323,1,2,5430292,0,5732887,0,0,1,10375,10369,11437,10714,0,0,0,0
2,2015-06-26 17:35:04.764434,1,2,5437169,0,5807836,0,1,1,10380,10369,10898,0,0,0,0,0
3,2015-06-26 17:36:30.324177,1,2,5467363,1,5684689,1,0,1,10380,10367,12780,12680,0,0,0,0
4,2015-06-26 17:40:26.117068,1,2,5442165,0,5781811,0,0,0,10380,10367,10708,10827,0,0,0,0


## 2. Apply Hensen Adapter

The adapter applies official filtering and mapping logic.

In [4]:
data_iterator = load_hensen_dataset(str(data_path))
df_filtered = pd.concat(list(data_iterator))
print(f"Valid Bell trial count: {len(df_filtered)}")

Valid Bell trial count: 245


## 3. Setting Pair Distribution

The 245 trials should be distributed across the four setting pairs (00, 01, 10, 11).

In [5]:
counts = df_filtered.groupby(['alice_setting', 'bob_setting']).size().reset_index(name='count')
print(counts)

   alice_setting  bob_setting  count
0              0            0     53
1              0            1     79
2              1            0     62
3              1            1     51


## 4. Run CHSH Validation

Verify the CHSH S-value matches the target $S \approx 2.42$.

In [6]:
results = run_open_data_chsh_validation(
    load_hensen_dataset(str(data_path)),
    dataset_name="hensen_audit",
    output_dir="../outputs/hensen_audit",
    bootstrap_samples=1000
)


--- Running Open Data CHSH Validation: hensen_audit ---
Phi_eff is an effective phenomenological parameter only. Without gravitational path, altitude, curvature, or spacetime-baseline metadata, this is not evidence of spacetime-induced X-Theta holonomy.
Calculating bootstrap (n=1000)...
Results saved to ../outputs/hensen_audit
S = 2.4225 ± 0.2038
Phi_eff = 0.4091


## 5. Conclusion

Target S: 2.42 ± 0.20.  
Observed S: {results['CHSH_S']:.4f} ± {results['CHSH_S_se']:.4f}.  
Valid Trials: {results['row_count']}.  

The reproduction is considered successful if valid trials $\approx 245$ and S is within tolerance.